In [1]:
from pathlib import Path
import getpass
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openrouter import ChatOpenRouter
from langchain_huggingface import HuggingFaceEmbeddings
import dotenv
print("-------ALL IMPORTS DONE-------")

C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_14968\3405200499.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders


-------ALL IMPORTS DONE-------


In [2]:
#loading the PDF file
path=r"D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf"
loader = PyPDFLoader(path)
pages = loader.load()
print(f"-------LOADED {len(pages)} PAGES FROM PDF-------")

-------LOADED 77 PAGES FROM PDF-------


In [3]:
def custom_metadata(page_number):
    if 1 <= page_number <= 2:
        return "front_matter"
    elif 3 <= page_number <= 4:
        return "introduction"
    elif 5 <= page_number <= 7:
        return "pretraining"
    elif 8 <= page_number <= 19:
        return "finetuning"
    elif 20 <= page_number <= 31:
        return "safety"
    elif 32 <= page_number <= 35:
        return "discussion"
    elif page_number == 36:
        return "conclusion"
    else:
        return "unknown"


In [ ]:
for page_document in pages:
    page_number = page_document.metadata["page"] + 1   # if pages start from 0
    page_document.metadata.update({
        "paper": "llama2-research-paper",
        "organization": "Meta",
        "year": "2023",
        "document_type": "research-paper",
        "section": custom_metadata(page_number),
        "access_level": "public"
    })

In [5]:
for page_document in pages[:5]:
    print(f"Page {page_document.metadata['page'] + 1} metadata: {page_document.metadata}")

Page 1 metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\GEN AI BASICS\\RAG\\Vector-database\\Data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'llama2-research-paper', 'organization': 'Meta', 'year': '2023', 'document_type': 'research-paper', 'section': 'front_matter', 'access_level': 'public'}
Page 2 metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/F

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
)
chunked_documents = text_splitter.split_documents(pages)
print(f"-------TOTAL PAGES: {len(pages)}-------")
print(f"-------SPLIT DOCUMENT INTO {len(chunked_documents)} CHUNKS-------")

-------TOTAL PAGES: 77-------
-------SPLIT DOCUMENT INTO 317 CHUNKS-------


In [34]:
for chunk_number, chunk in enumerate(chunked_documents):
    page_number=chunk.metadata.get("page_number", "unknown")
    chunk.metadata['chunkid']=(f"llama2-research-paper_page{page_number}_chunk{chunk_number}")

In [35]:
print("Chunk content:")
print(chunked_documents[0].page_content[:1000])

print("\nChunk metadata:")
print(chunked_documents[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
dimensions = embeddings_hf.embed_query("Hello, world!")
print(f"-------EMBEDDINGS DIMENSIONS: {len(dimensions)}-------")

-------EMBEDDINGS DIMENSIONS: 384-------


In [26]:
vectorstore = Chroma(
    collection_name="llama2-research-paper",
    embedding_function=embeddings_hf,
    persist_directory="D:/GEN AI BASICS/RAG/Vector-database/Data/chroma_db",
    collection_metadata={
            "hnsw:space": "cosine"
        }

)
print("-------VECTORSTORE CREATED-------")
print(f"stored chunks{len(chunked_documents)}")
documents = vectorstore.add_documents(chunked_documents)

-------VECTORSTORE CREATED-------
stored chunks317


In [28]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [29]:
query="How does LLaMA 2 ensure safety in its models?"
similarity_docs = similarity_retriever.invoke(query)